In [134]:
import requests
from shapely.geometry import Point, shape
import shapely
import geojson
import json
from pprint import pprint
from shapely.ops import unary_union

import pickle


OVERPASS_URL = "https://overpass-api.de/api/interpreter"

# Countries

### Extract borders of all countries

In [135]:
with open('ALL-high-res.json', encoding='utf-8') as f:
    data = json.load(f)

In [143]:
result = {}
for feature in data["features"]:
    iso = feature["properties"]["iso_a2_eh"]

    if iso == '-99':
        continue

    if iso not in result:
        result[iso] = {
            "name": None,
            "geoms": [],
            "poly": None
        }

    name = feature["properties"]["name_ciawf"]
    if result[iso]["name"] is None and name is not None:
        result[iso]["name"] = name

    poly = shapely.from_geojson(json.dumps(feature["geometry"]))
    result[iso]["geoms"].append(poly)

for key, value in result.items():
    if len(value["geoms"]) > 0:
        value["poly"] = unary_union(value["geoms"])
    else:
        value["poly"] = value["geoms"]

with open("countries.pkl", "wb") as f:
    pickle.dump(result, f)


# OpenStreetmap

## Fetch cities

In [ ]:
query = """
[out:json][timeout:180];
(
  node[place~"city|town|village"](around:30000,49.440358,6.07191);
);
out center;
"""

# query = """
# [out:json][timeout:180];
# (
#   node[place~"city|town|village"](around:300000,49.129785,6.168549);
# );
# out center;
# """
response = requests.post(OVERPASS_URL, data={"data": query})
data = response.json()

cities = [
    {
        "name": elem["tags"].get("name", "unknown"),
        "type": elem["tags"].get("place"),
        "lat": elem["lat"],
        "lon": elem["lon"]
    }
    for elem in data["elements"]
]

for city in cities[:10]:
    print(city)

{'name': 'Charleroi', 'type': 'city', 'lat': 50.4116233, 'lon': 4.444528}
{'name': 'Raidelbach', 'type': 'village', 'lat': 49.709349, 'lon': 8.7352499}
{'name': 'Reichenbach', 'type': 'village', 'lat': 49.7129657, 'lon': 8.6930453}
{'name': 'Elmshausen', 'type': 'village', 'lat': 49.7016047, 'lon': 8.6722226}
{'name': 'Wilmshausen', 'type': 'village', 'lat': 49.6957672, 'lon': 8.6620534}
{'name': 'Schönberg', 'type': 'village', 'lat': 49.6943101, 'lon': 8.6489156}
{'name': 'Bensheim', 'type': 'town', 'lat': 49.6810158, 'lon': 8.6227577}
{'name': 'Wendlingen am Neckar', 'type': 'town', 'lat': 48.6727602, 'lon': 9.3838404}
{'name': 'Esslingen am Neckar', 'type': 'town', 'lat': 48.7427584, 'lon': 9.3071685}
{'name': 'Plochingen', 'type': 'town', 'lat': 48.711173, 'lon': 9.4184961}


## Include country

In [155]:
from shapely.geometry import Point
from shapely.strtree import STRtree

# 1. Vos données (exemples)
country_name, country_poly = [], []
with open("countries.pkl", "rb") as f:
    result = pickle.load(f)
for country in result.values():
    country_name.append(country["name"])
    country_poly.append(country["poly"])

# 2. Créer l'index spatial sur les pays
tree = STRtree(country_poly)

# 3. Créer les points Shapely
points = [Point(city["lon"], city["lat"]) for city in cities]

# 4. Requête de masse : qui est à l'intérieur de quoi ?
# query_bulk retourne deux tableaux d'indices : [indices_points, indices_pays]
indices_points, indices_pays = tree.query(points, predicate='within')

# 5. Associer les résultats
for idx_pt, idx_country in zip(indices_points, indices_pays):
    cities[idx_pt]["country"] = country_name[idx_country]

In [156]:
for city in cities:
    if "udange" in city["name"].lower():
        print(city["name"])

Udange
Azoudange
Haboudange
Hombourg-Budange
Budange


In [158]:
with open("cities.pkl", "wb") as f:
    pickle.dump(result, f)